# 06 — Experimentos A/B
Compara a Regressão Logística usando somente contexto (A) versus contexto + dinâmica do acidente (B). O teste de 2025 não é utilizado.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import joblib
ROOT=Path.cwd(); OUT=ROOT/"outputs"; DATA=ROOT/"data"/"processed"
SILVER=DATA/"prf_acidentes_silver_2020_2025.parquet"
for p in ["modelos","interpretabilidade","modelo_final"]:
    (OUT/p).mkdir(parents=True,exist_ok=True)
df=pd.read_parquet(SILVER)
df["br_categoria"]=pd.to_numeric(df.br,errors="coerce").astype("Int64").astype("string").fillna("Não informado")
flags=sorted(c for c in df if c.startswith("tracado_") and c!="tracado_via")
categoricas=["uf","br_categoria","dia_semana","fase_dia","sentido_via","condicao_metereologica","tipo_pista","uso_solo","tipo_acidente"]
numericas=["km","mes","hora","fim_semana","qtd_caracteristicas_tracado","veiculos"]+flags
features=categoricas+numericas
proibidas={"pessoas","mortos","feridos_graves","feridos_leves","feridos","ilesos","ignorados","classificacao_acidente","acidente_fatal"}
assert set(features).isdisjoint(proibidas)
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
def pre(cat=categoricas,num=numericas):
    try: enc=OneHotEncoder(handle_unknown="ignore",min_frequency=100,sparse_output=True)
    except TypeError: enc=OneHotEncoder(handle_unknown="ignore",min_frequency=100,sparse=True)
    cat_pipe=Pipeline([("imputar",SimpleImputer(strategy="most_frequent")),("onehot",enc)])
    num_pipe=Pipeline([("imputar",SimpleImputer(strategy="median")),("escala",StandardScaler())])
    return ColumnTransformer([("categoricas",cat_pipe,cat),("numericas",num_pipe,num)])
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,fbeta_score,roc_auc_score,average_precision_score,confusion_matrix
def avaliar(y,proba,nome,limiar=0.5):
    pred=(proba>=limiar).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return dict(modelo=nome,threshold=limiar,accuracy=accuracy_score(y,pred),
      precision=precision_score(y,pred,zero_division=0),recall=recall_score(y,pred,zero_division=0),
      f1=f1_score(y,pred,zero_division=0),f2=fbeta_score(y,pred,beta=2,zero_division=0),
      roc_auc=roc_auc_score(y,proba),pr_auc=average_precision_score(y,proba),
      verdadeiro_negativo=tn,falso_positivo=fp,falso_negativo=fn,verdadeiro_positivo=tp)


In [ ]:
from sklearn.linear_model import LogisticRegression
cat_a=[c for c in categoricas if c!="tipo_acidente"]
num_a=[c for c in numericas if c!="veiculos"]
cat_b=categoricas;num_b=numericas
treino=df[df.ano<=2023];valid=df[df.ano==2024]
linhas=[]
for nome,ca,nu in [("Modelo A - Contexto",cat_a,num_a),("Modelo B - Contexto e dinâmica",cat_b,num_b)]:
    ff=ca+nu
    clf=Pipeline([("preprocessamento",pre(ca,nu)),("modelo",LogisticRegression(solver="saga",max_iter=500,random_state=42))])
    clf.fit(treino[ff],treino.acidente_grave)
    prob=clf.predict_proba(valid[ff])[:,1]
    linhas.append(avaliar(valid.acidente_grave,prob,nome))
    joblib.dump(clf,OUT/"modelos"/(nome.split()[1].lower()+"_logistica.joblib"))
res=pd.DataFrame(linhas)
display(res)
res.to_csv(OUT/"modelos"/"baseline_logistic_regression.csv",index=False)